# 02 - Exploratory Data Analysis (EDA)
## EV Charging Station Congestion & Grid Load Predictor

This notebook performs a comprehensive exploratory analysis of the processed 2,000-row dataset, uncovering spatial, temporal, energy, and congestion distribution characteristics across the EV charging network.


### 1. Load Processed Dataset


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual theme
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

data_path = Path("../data/processed/ev_charging_project_2000.csv")
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape}")
df.info()


### 2. Summary Statistics


In [ ]:
df.describe().round(2)


### 3. Charging Sessions by Station (Top 15 Most Active Stations)


In [ ]:
top_stations = df["station_id"].value_counts().head(15)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_stations.index.astype(str), y=top_stations.values, color="#1f77b4")
plt.title("Top 15 Most Active Charging Stations (Session Count)", fontsize=13, fontweight="bold")
plt.xlabel("Station ID")
plt.ylabel("Session Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 4. Charging Sessions by Hour of Day


In [ ]:
hourly_sessions = df["hour"].value_counts().sort_index()
plt.figure(figsize=(12, 5))
sns.barplot(x=hourly_sessions.index, y=hourly_sessions.values, color="#2ca02c")
plt.title("Charging Session Distribution by Hour of the Day", fontsize=13, fontweight="bold")
plt.xlabel("Hour (24-hour clock)")
plt.ylabel("Number of Sessions")
plt.tight_layout()
plt.show()


### 5. Charging Sessions by Day of Week


In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="day_of_week", order=day_order, color="#ff7f0e")
plt.title("Weekly Session Distribution Across Days", fontsize=13, fontweight="bold")
plt.xlabel("Day of Week")
plt.ylabel("Session Count")
plt.tight_layout()
plt.show()


### 6. Energy Consumption & Charging Duration Distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy histogram
sns.histplot(df["energy_kwh"], bins=30, kde=True, ax=axes[0], color="#9467bd")
axes[0].set_title("Energy Consumption (kWh) Distribution", fontweight="bold")
axes[0].set_xlabel("Energy (kWh)")
axes[0].set_ylabel("Frequency")

# Charging duration histogram
sns.histplot(df["charge_time_hrs"], bins=30, kde=True, ax=axes[1], color="#8c564b")
axes[1].set_title("Charging Duration (Hours) Distribution", fontweight="bold")
axes[1].set_xlabel("Duration (Hours)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


### 7. Congestion Level and Grid Load Target Distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tier_order = ["Low", "Medium", "High"]
palette = {"Low": "#2ca02c", "Medium": "#ff7f0e", "High": "#d62728"}

sns.countplot(data=df, x="congestion_level", order=tier_order, palette=palette, ax=axes[0])
axes[0].set_title("Congestion Level Target Distribution", fontweight="bold")
axes[0].set_xlabel("Congestion Tier")
axes[0].set_ylabel("Session Count")

sns.countplot(data=df, x="load_level", order=tier_order, palette=palette, ax=axes[1])
axes[1].set_title("Grid Load Level Target Distribution", fontweight="bold")
axes[1].set_xlabel("Grid Load Tier")
axes[1].set_ylabel("Session Count")

plt.tight_layout()
plt.show()


### 8. Hour vs Congestion Level


In [ ]:
hour_c = pd.crosstab(df["hour"], df["congestion_level"], normalize="index")[tier_order]
plt.figure(figsize=(14, 6))
hour_c.plot(kind="bar", stacked=True, color=["#2ca02c", "#ff7f0e", "#d62728"], figsize=(14, 6))
plt.title("Proportion of Congestion Levels by Hour of the Day", fontsize=13, fontweight="bold")
plt.xlabel("Hour")
plt.ylabel("Proportion")
plt.legend(title="Congestion Tier")
plt.tight_layout()
plt.show()


### 9. Top Stations vs Congestion Breakdown


In [ ]:
top_10 = df["station_id"].value_counts().head(10).index
station_c = pd.crosstab(df[df["station_id"].isin(top_10)]["station_id"], df["congestion_level"])[tier_order]
station_c.plot(kind="bar", stacked=True, color=["#2ca02c", "#ff7f0e", "#d62728"], figsize=(12, 5))
plt.title("Congestion Distribution in Top 10 Stations", fontsize=13, fontweight="bold")
plt.xlabel("Station ID")
plt.ylabel("Session Count")
plt.legend(title="Congestion Tier")
plt.tight_layout()
plt.show()


### 10. Key Insights & Findings
1. **Diurnal Peaks**: Charging sessions surge from 8:00 AM to 11:00 AM and again in late afternoon (4:00 PM to 7:00 PM), reflecting workplace arrival and evening commute patterns.
2. **Balanced Target Representation**: Derived academic proxies for Congestion Level and Grid Load achieve balanced representation across `Low`, `Medium`, and `High` categories (~33% each).
3. **Station Heterogeneity**: Several primary stations exhibit consistent `High` congestion during commute windows, confirming the need for an automated alternative station recommendation mechanism.
